# Wang-Sheeley-Arge (WSA) Solar Wind Prediction Model — Implementation / 구현

**Paper**: Arge, C. N. and V. J. Pizzo (2000), "Improvement in the prediction of solar wind conditions using near-real time solar magnetic field updates," *J. Geophys. Res.*, 105(A5), 10465–10479.

This notebook implements the key algorithms from the WSA model paper:
이 노트북은 WSA 모델 논문의 핵심 알고리즘을 구현합니다:

1. **Expansion factor–velocity relation** $v(f_s)$ / 팽창 계수–속도 관계식 (Eq. 4)
2. **Magnetogram quality factor** $Q$ / Magnetogram 품질 계수 (Eq. 1)
3. **LOS to radial field correction** / 시선 방향 → 방사 자기장 보정 (Eqs. 2-3)
4. **Stream interaction propagation** / 스트림 상호작용 전파 (Eq. 5)
5. **Simplified PFSS with dipole approximation** / 간소화된 PFSS (쌍극자 근사)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from scipy.special import lpmv  # Associated Legendre polynomials

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# Physical constants
R_SUN = 6.96e10       # Solar radius [cm]
R_SS = 2.5            # Source surface radius [R_sun]
AU_CM = 1.496e13      # 1 AU [cm]
OMEGA_SUN = 2 * np.pi / (25.4 * 86400)  # Sidereal rotation rate [rad/s]

print("WSA Model Implementation — Arge & Pizzo (2000)")
print(f"Source surface: {R_SS} R_sun")
print(f"Solar radius: {R_SUN:.2e} cm")

## Part 1: Expansion Factor–Velocity Relation / 팽창 계수–속도 관계

The core empirical relation of the WSA model (Eq. 4) maps the flux tube expansion factor $f_s$ to solar wind velocity at the source surface:

WSA 모델의 핵심 경험 관계식 (Eq. 4)은 flux tube expansion factor $f_s$를 source surface에서의 태양풍 속도로 매핑합니다:

$$v(f_s) = 267.5 + \frac{410}{(f_s)^{2/5}} \quad \text{[km/s]}$$

Key difference from original Wang-Sheeley: this velocity is assigned **at the source surface**, not at Earth.

기존 Wang-Sheeley와의 핵심 차이: 이 속도는 지구가 아닌 **source surface에서** 지정됩니다.

In [ ]:
def wsa_velocity(fs: np.ndarray) -> np.ndarray:
    """Compute solar wind velocity from expansion factor (Eq. 4).

    Args:
        fs: Flux tube expansion factor(s). Must be >= 1.

    Returns:
        Solar wind velocity at source surface [km/s].
    """
    return 267.5 + 410.0 / fs ** 0.4


# --- Plot v(f_s) relation ---
fs_vals = np.linspace(1, 150, 500)
v_vals = wsa_velocity(fs_vals)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Linear scale
ax1.plot(fs_vals, v_vals, 'b-', linewidth=2)
ax1.set_xlabel('Expansion Factor $f_s$')
ax1.set_ylabel('Solar Wind Speed $v$ [km/s]')
ax1.set_title('WSA $v(f_s)$ Relation (Eq. 4) — Linear Scale\nWSA $v(f_s)$ 관계식 — 선형 스케일')
ax1.axhline(y=267.5, color='r', linestyle='--', alpha=0.5, label='$v_{min}$ = 267.5 km/s')
ax1.axhline(y=677.5, color='g', linestyle='--', alpha=0.5, label='$v_{max}$ ($f_s$=1) = 677.5 km/s')
ax1.legend()

# Log scale for f_s
ax2.semilogx(fs_vals, v_vals, 'b-', linewidth=2)
ax2.set_xlabel('Expansion Factor $f_s$ (log scale)')
ax2.set_ylabel('Solar Wind Speed $v$ [km/s]')
ax2.set_title('WSA $v(f_s)$ Relation — Log Scale\nWSA $v(f_s)$ 관계식 — 로그 스케일')

# Annotate physical regions
ax2.axvspan(1, 5, alpha=0.1, color='blue', label='Coronal hole center / CH 중심')
ax2.axvspan(5, 30, alpha=0.1, color='green', label='CH boundary / CH 경계')
ax2.axvspan(30, 150, alpha=0.1, color='red', label='Streamer belt / 스트리머 벨트')
ax2.legend(loc='upper right', fontsize=9)

plt.tight_layout()
plt.show()

# Print table of representative values
print("\n=== Representative v(f_s) values / 대표 v(f_s) 값 ===")
print(f"{'f_s':>6s}  {'v [km/s]':>10s}  {'Region':>25s}")
print("-" * 50)
for fs, region in [(1, 'CH center / CH 중심'),
                   (3, 'Deep CH / CH 내부'),
                   (10, 'CH boundary / CH 경계'),
                   (30, 'Near streamer / 스트리머 근처'),
                   (100, 'Closed field / 폐쇄 자기장')]:
    print(f"{fs:>6d}  {wsa_velocity(fs):>10.1f}  {region:>25s}")

## Part 2: Magnetogram Quality Factor $Q$ / Magnetogram 품질 계수

The quality factor $Q$ (Eq. 1) evaluates individual magnetograms by comparing their polar field properties against a reference set of 62 "good" magnetograms:

품질 계수 $Q$ (Eq. 1)는 개별 magnetogram의 극 자기장 특성을 62개의 "good" 참조 magnetogram과 비교하여 품질을 평가합니다:

$$Q = \sqrt{\left(\frac{\langle B \rangle - \langle B_\text{ref} \rangle}{\langle \sigma \rangle_{B(\text{ref})}}\right)^2 + \left(\frac{\sigma_B}{\sigma_{B(\text{ref})}}\right)^2} \times \left(\frac{|B_\text{max} - B_\text{min}|}{\langle \sigma \rangle_{B(\text{ref})}}\right)$$

- $Q < 6.5$: Accept / 수용
- $Q \geq 6.5$: Trim edges or reject / 가장자리 제거 또는 폐기

In [ ]:
def compute_q_factor(
    B_mean: float,
    sigma_B: float,
    B_max: float,
    B_min: float,
    B_ref_mean: float,
    sigma_B_ref: float,
    sigma_mean_ref: float,
) -> float:
    """Compute magnetogram quality factor Q (Eq. 1).

    Args:
        B_mean: Mean polar field of the magnetogram [μT].
        sigma_B: Standard deviation of polar field [μT].
        B_max: Maximum polar field value [μT].
        B_min: Minimum polar field value [μT].
        B_ref_mean: Mean polar field of reference set [μT].
        sigma_B_ref: Std dev of reference set's mean polar fields [μT].
        sigma_mean_ref: Mean of reference set's std devs [μT].

    Returns:
        Quality factor Q. Values < 6.5 indicate good quality.
    """
    term1 = ((B_mean - B_ref_mean) / sigma_mean_ref) ** 2
    term2 = (sigma_B / sigma_B_ref) ** 2
    term3 = abs(B_max - B_min) / sigma_mean_ref
    return np.sqrt(term1 + term2) * term3


def screen_magnetogram(Q_north: float, Q_south: float,
                       trim_step: float = 5.0,
                       max_trim: float = 25.0,
                       threshold: float = 6.5) -> str:
    """Determine magnetogram disposition based on Q factors.

    Args:
        Q_north: Q factor for north pole.
        Q_south: Q factor for south pole.
        trim_step: Degrees to trim from each edge per iteration.
        max_trim: Maximum total trim before rejection.
        threshold: Q factor acceptance threshold.

    Returns:
        Disposition string: 'accept', 'trim_Xdeg', or 'reject'.
    """
    if Q_north < threshold and Q_south < threshold:
        return 'accept'
    # Simulate iterative trimming
    for trim in np.arange(trim_step, max_trim + trim_step, trim_step):
        # In reality, Q would be recomputed after trimming.
        # Here we model Q decreasing ~linearly with trimming.
        Q_n_trimmed = Q_north * (1 - 0.15 * trim / trim_step)
        Q_s_trimmed = Q_south * (1 - 0.15 * trim / trim_step)
        if Q_n_trimmed < threshold and Q_s_trimmed < threshold:
            return f'trim_{trim:.0f}deg'
    return 'reject'


# --- Simulate reference set and test magnetograms ---
np.random.seed(42)

# Simulated reference set properties (based on paper: ~100-150 μT polar fields)
B_ref_mean_north = 120.0   # μT
sigma_B_ref_north = 25.0   # μT (spread across 62 reference magnetograms)
sigma_mean_ref_north = 30.0  # μT

# Generate synthetic test magnetograms
n_test = 200
B_means = np.random.normal(B_ref_mean_north, 40, n_test)
sigma_Bs = np.abs(np.random.normal(sigma_mean_ref_north, 15, n_test))
B_spreads = np.abs(np.random.normal(100, 60, n_test))  # |Bmax - Bmin|

Q_values = np.array([
    compute_q_factor(bm, sb, bm + bs/2, bm - bs/2,
                     B_ref_mean_north, sigma_B_ref_north, sigma_mean_ref_north)
    for bm, sb, bs in zip(B_means, sigma_Bs, B_spreads)
])

# Classify
accepted = Q_values < 6.5
rejected = Q_values >= 6.5

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Q factor distribution
ax1.hist(Q_values[accepted], bins=20, alpha=0.7, color='green', label=f'Accept ({accepted.sum()})')
ax1.hist(Q_values[rejected], bins=20, alpha=0.7, color='red', label=f'Reject/Trim ({rejected.sum()})')
ax1.axvline(x=6.5, color='k', linestyle='--', linewidth=2, label='Threshold Q=6.5')
ax1.set_xlabel('Quality Factor $Q$')
ax1.set_ylabel('Count / 개수')
ax1.set_title('Magnetogram Quality Distribution\nMagnetogram 품질 분포')
ax1.legend()

# Q vs B_spread (key diagnostic)
ax2.scatter(B_spreads[accepted], Q_values[accepted], alpha=0.5, c='green',
            s=20, label='Accept')
ax2.scatter(B_spreads[rejected], Q_values[rejected], alpha=0.5, c='red',
            s=20, label='Reject/Trim')
ax2.axhline(y=6.5, color='k', linestyle='--', linewidth=2)
ax2.set_xlabel('Polar Field Spread $|B_{max} - B_{min}|$ [μT]')
ax2.set_ylabel('Quality Factor $Q$')
ax2.set_title('$Q$ vs Polar Field Spread\n$Q$ 대 극 자기장 산포')
ax2.legend()

plt.tight_layout()
plt.show()

pct_rejected = rejected.sum() / n_test * 100
print(f"\nRejection rate: {pct_rejected:.1f}% (paper reports ~12% discarded)")
print(f"Accepted: {accepted.sum()}/{n_test}, Rejected/Trimmed: {rejected.sum()}/{n_test}")

## Part 3: LOS to Radial Field Correction / 시선 방향 → 방사 자기장 보정

WSO magnetograms measure the line-of-sight (LOS) component $B_l$, which must be corrected to obtain the radial field $B_r$. Under the assumption that the photospheric field is purely radial (Eqs. 2-3):

WSO magnetogram은 시선 방향(LOS) 성분 $B_l$을 측정하며, 방사 자기장 $B_r$를 얻으려면 보정이 필요합니다. 광구 자기장이 순수 방사형이라는 가정 하에 (Eqs. 2-3):

$$B_l = B_r \sin\theta \cos(\phi - \phi_o)$$

Therefore: $B_r = B_l / [\sin\theta \cos(\phi - \phi_o)]$

This correction diverges at the limb ($\phi - \phi_o \to \pm 90°$) and poles ($\theta \to 0°, 180°$), which is why edge data is unreliable.

이 보정은 limb ($\phi - \phi_o \to \pm 90°$)과 극 ($\theta \to 0°, 180°$)에서 발산하므로, 가장자리 데이터가 신뢰할 수 없습니다.

In [ ]:
def los_to_radial(B_los: np.ndarray,
                  theta: np.ndarray,
                  delta_phi: np.ndarray) -> np.ndarray:
    """Convert LOS magnetic field to radial field (Eq. 3).

    Args:
        B_los: Line-of-sight magnetic field [μT].
        theta: Colatitude [radians].
        delta_phi: Longitude offset from central meridian [radians].

    Returns:
        Radial magnetic field B_r [μT].
    """
    projection = np.sin(theta) * np.cos(delta_phi)
    # Avoid division by zero near limb/poles
    projection = np.where(np.abs(projection) < 0.1, np.nan, projection)
    return B_los / projection


def apply_b_angle_correction(B_polar: float,
                             b_angle: float,
                             hemisphere: str = 'north') -> float:
    """Apply solar b angle correction to polar field measurements.

    The Sun's rotation axis is tilted 7.25° to the ecliptic, causing
    annual variations in polar field visibility.

    Args:
        B_polar: Measured polar field [μT].
        b_angle: Solar b angle [degrees], range [-7.25, +7.25].
        hemisphere: 'north' or 'south'.

    Returns:
        Corrected polar field [μT].
    """
    b_rad = np.radians(b_angle)
    if hemisphere == 'north':
        # North pole better observed when b > 0
        correction = 1.0 + 0.3 * np.sin(b_rad)  # Simplified model
    else:
        # South pole better observed when b < 0
        correction = 1.0 - 0.3 * np.sin(b_rad)
    return B_polar / correction


# --- Demonstrate projection effects ---
theta_vals = np.linspace(np.radians(15), np.radians(165), 100)  # Colatitude
dphi_vals = np.linspace(np.radians(-55), np.radians(55), 100)   # ±55° from CM

theta_grid, dphi_grid = np.meshgrid(theta_vals, dphi_vals)
projection = np.sin(theta_grid) * np.cos(dphi_grid)
# Correction factor = 1/projection
correction_factor = 1.0 / projection

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Projection factor map
im0 = axes[0].pcolormesh(np.degrees(dphi_vals), np.degrees(theta_vals),
                          projection.T, cmap='RdYlBu_r', vmin=0, vmax=1)
axes[0].set_xlabel('$\\phi - \\phi_o$ [deg]')
axes[0].set_ylabel('Colatitude $\\theta$ [deg]')
axes[0].set_title('Projection Factor $\\sin\\theta \\cos(\\phi-\\phi_o)$\n투영 계수')
plt.colorbar(im0, ax=axes[0])

# Correction factor map (capped at 5)
corr_capped = np.clip(correction_factor, 1, 5)
im1 = axes[1].pcolormesh(np.degrees(dphi_vals), np.degrees(theta_vals),
                          corr_capped.T, cmap='hot', vmin=1, vmax=5)
axes[1].set_xlabel('$\\phi - \\phi_o$ [deg]')
axes[1].set_ylabel('Colatitude $\\theta$ [deg]')
axes[1].set_title('Correction Factor $1/[\\sin\\theta \\cos(\\phi-\\phi_o)]$\n보정 계수 (capped at 5)')
plt.colorbar(im1, ax=axes[1], label='Correction factor')

# Solar b angle effect on polar fields
b_angles = np.linspace(-7.25, 7.25, 100)
B_true_north = 120.0  # True north polar field
B_true_south = -110.0  # True south polar field

B_meas_north = [B_true_north * (1 + 0.3 * np.sin(np.radians(b))) for b in b_angles]
B_meas_south = [B_true_south * (1 - 0.3 * np.sin(np.radians(b))) for b in b_angles]

axes[2].plot(b_angles, B_meas_north, 'b-', linewidth=2, label='North Pole / 북극')
axes[2].plot(b_angles, np.abs(B_meas_south), 'r-', linewidth=2, label='|South Pole| / |남극|')
axes[2].axhline(y=B_true_north, color='b', linestyle='--', alpha=0.4)
axes[2].axhline(y=abs(B_true_south), color='r', linestyle='--', alpha=0.4)
axes[2].set_xlabel('Solar $b$ angle [deg]')
axes[2].set_ylabel('Measured Polar Field [μT]')
axes[2].set_title('Solar $b$ Angle Effect on Polar Fields\nSolar $b$ angle이 극 자기장에 미치는 영향')
axes[2].legend()

plt.tight_layout()
plt.show()

# Numerical example
print("=== LOS Correction Example / LOS 보정 예시 ===")
B_los_example = 50.0  # μT
for angle_deg in [0, 20, 40, 55]:
    angle_rad = np.radians(angle_deg)
    theta_eq = np.radians(90)  # Solar equator
    B_r = los_to_radial(B_los_example, theta_eq, angle_rad)
    factor = 1.0 / (np.sin(theta_eq) * np.cos(angle_rad))
    print(f"  Δφ = {angle_deg:>3d}° → B_r = {B_r:>7.1f} μT "
          f"(correction factor: {factor:.2f}x)")

## Part 4: Stream Interaction Propagation / 스트림 상호작용 전파

The WSA model propagates solar wind from the source surface (2.5 $R_\odot$) to Earth (1 AU) in 1/8 AU steps. When a faster element catches a slower one, they merge using a near-harmonic-mean weighting (Eq. 5):

WSA 모델은 태양풍을 source surface (2.5 $R_\odot$)에서 지구 (1 AU)까지 1/8 AU 단위로 전파합니다. 빠른 요소가 느린 요소를 따라잡으면 조화 평균에 가까운 가중 함수로 병합합니다 (Eq. 5):

$$v_i = \sqrt{\frac{2}{1/v_i^2 + 1/v_{i+1}^2}}$$

This is the **quadratic harmonic mean** — it weights slower speeds more heavily, physically reflecting that when a fast stream runs into a slow one, the resulting merged speed is closer to the slow stream's speed.

이것은 **이차 조화 평균**으로, 느린 속도에 더 큰 가중치를 부여합니다. 물리적으로 빠른 스트림이 느린 스트림에 충돌할 때, 결과 속도가 느린 쪽에 더 가까워지는 것을 반영합니다.

In [ ]:
def stream_merge_velocity(v1: float, v2: float) -> float:
    """Compute merged velocity of two interacting streams (Eq. 5).

    Args:
        v1: Velocity of first stream element [km/s].
        v2: Velocity of second stream element [km/s].

    Returns:
        Merged velocity [km/s].
    """
    return np.sqrt(2.0 / (1.0 / v1**2 + 1.0 / v2**2))


def propagate_to_earth(velocities: np.ndarray,
                       release_times: np.ndarray,
                       dr_au: float = 0.125) -> tuple:
    """Propagate solar wind elements from source surface to 1 AU.

    Each element travels radially at its assigned velocity. When faster
    elements catch slower ones, they merge using the weighting function.

    Args:
        velocities: Initial velocities at source surface [km/s].
        release_times: Release times of each element [days].
        dr_au: Radial step size [AU]. Default 1/8 AU.

    Returns:
        Tuple of (arrival_times [days], arrival_velocities [km/s]).
    """
    n = len(velocities)
    v = velocities.copy()
    t = release_times.copy()

    dr_km = dr_au * 1.496e8  # Convert AU to km
    n_steps = int(round(1.0 / dr_au))  # Steps to reach 1 AU

    for step in range(n_steps):
        # Advance each element by dr at its current velocity
        dt = dr_km / v  # Time to travel dr [seconds]
        t = t + dt / 86400.0  # Convert to days

        # Check for stream interactions: if element i arrives
        # after element i+1 (faster behind catches slower ahead),
        # merge them.
        i = 0
        while i < len(t) - 1:
            if t[i] >= t[i + 1]:
                # Merge: faster element caught slower one
                v_merged = stream_merge_velocity(v[i], v[i + 1])
                v[i] = v_merged
                t[i] = t[i + 1]  # Adjust to earlier arrival
                # Remove the caught element
                v = np.delete(v, i + 1)
                t = np.delete(t, i + 1)
            else:
                i += 1

    return t, v


# --- Demonstrate stream propagation with synthetic data ---
np.random.seed(123)

# Create a synthetic ecliptic velocity profile:
# 72 cells × 5° = 360° of Carrington longitude
n_cells = 72
longitudes = np.linspace(0, 360, n_cells, endpoint=False)

# Synthetic expansion factors: two coronal holes (low f_s) + slow wind regions
fs_profile = np.ones(n_cells) * 30.0  # Base: slow wind
# Coronal hole 1: centered at 90°
ch1_center, ch1_width = 90, 30
mask1 = np.abs(longitudes - ch1_center) < ch1_width
fs_profile[mask1] = 2.0 + 3.0 * (np.abs(longitudes[mask1] - ch1_center) / ch1_width)
# Coronal hole 2: centered at 250°
ch2_center, ch2_width = 250, 20
mask2 = np.abs(longitudes - ch2_center) < ch2_width
fs_profile[mask2] = 3.0 + 5.0 * (np.abs(longitudes[mask2] - ch2_center) / ch2_width)

# Convert to velocities at source surface
v_source = wsa_velocity(fs_profile)

# Release times: Sun rotates ~13.2°/day, so 5° ≈ 0.379 days apart
release_times = longitudes / 13.2  # days from start

# Propagate to Earth
t_earth, v_earth = propagate_to_earth(v_source, release_times)

# --- Plot results ---
fig = plt.figure(figsize=(14, 10))
gs = GridSpec(2, 2, figure=fig)

# Top left: Expansion factor profile
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(longitudes, fs_profile, 'k-', linewidth=2)
ax1.set_xlabel('Carrington Longitude [deg]')
ax1.set_ylabel('Expansion Factor $f_s$')
ax1.set_title('Synthetic Expansion Factor Profile\n합성 팽창 계수 프로파일')
ax1.set_yscale('log')
ax1.axhspan(1, 5, alpha=0.1, color='blue', label='CH')
ax1.legend()

# Top right: Velocity at source surface
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(longitudes, v_source, 'b-', linewidth=2, label='Source surface')
ax2.set_xlabel('Carrington Longitude [deg]')
ax2.set_ylabel('Velocity [km/s]')
ax2.set_title('Velocity at Source Surface\nSource Surface에서의 속도')
ax2.legend()

# Bottom left: Velocity time series comparison
ax3 = fig.add_subplot(gs[1, 0])
ax3.plot(release_times, v_source, 'b--', linewidth=1.5,
         alpha=0.7, label='Source surface (before propagation)')
ax3.plot(t_earth, v_earth, 'r-', linewidth=2,
         label='At Earth (after propagation)')
ax3.set_xlabel('Time [days]')
ax3.set_ylabel('Velocity [km/s]')
ax3.set_title('Velocity: Source Surface vs Earth\n속도: Source Surface 대 지구')
ax3.legend()

# Bottom right: Compare weighting functions
ax4 = fig.add_subplot(gs[1, 1])
v_slow = 350  # km/s
v_fast_range = np.linspace(350, 700, 100)
v_arithmetic = (v_slow + v_fast_range) / 2
v_harmonic = 2 / (1/v_slow + 1/v_fast_range)
v_wsa = np.sqrt(2 / (1/v_slow**2 + 1/v_fast_range**2))

ax4.plot(v_fast_range, v_arithmetic, 'g-', linewidth=2, label='Arithmetic mean')
ax4.plot(v_fast_range, v_wsa, 'r-', linewidth=2, label='WSA (Eq. 5)')
ax4.plot(v_fast_range, v_harmonic, 'b--', linewidth=2, label='Harmonic mean')
ax4.axhline(y=v_slow, color='gray', linestyle=':', alpha=0.5)
ax4.set_xlabel('Fast Stream $v_{fast}$ [km/s]')
ax4.set_ylabel('Merged Velocity [km/s]')
ax4.set_title(f'Merging Functions ($v_{{slow}}$ = {v_slow} km/s)\n병합 함수 비교')
ax4.legend()

plt.tight_layout()
plt.show()

# Statistics
print("=== Propagation Results / 전파 결과 ===")
print(f"Elements at source surface: {n_cells}")
print(f"Elements at Earth (after merging): {len(v_earth)}")
print(f"Elements merged: {n_cells - len(v_earth)}")
print(f"Velocity range at source: {v_source.min():.0f}–{v_source.max():.0f} km/s")
print(f"Velocity range at Earth:  {v_earth.min():.0f}–{v_earth.max():.0f} km/s")
travel_time = t_earth.mean() - release_times.mean()
print(f"Mean travel time: {travel_time:.1f} days (paper: ~4 days)")

## Part 5: Simplified PFSS with Dipole Approximation / 간소화된 PFSS (쌍극자 근사)

The full PFSS model solves Laplace's equation $\nabla^2 \Phi = 0$ between $R_\odot$ and $R_{ss}$ using spherical harmonics up to $\ell = 30$. Here we implement a simplified version using only the dipole term ($\ell = 1$) to illustrate the key concepts:

전체 PFSS 모델은 $R_\odot$에서 $R_{ss}$ 사이에서 Laplace 방정식을 $\ell = 30$까지의 구면 조화 함수로 풉니다. 여기서는 쌍극자 항($\ell = 1$)만 사용하는 간소화된 버전을 구현하여 핵심 개념을 설명합니다:

1. Compute the potential field from a dipole source / 쌍극자 소스에서 포텐셜장 계산
2. Trace field lines from source surface to photosphere / Source surface에서 광구까지 자기력선 추적
3. Calculate expansion factors along open field lines / 열린 자기력선에서 팽창 계수 계산
4. Map to solar wind velocity using $v(f_s)$ / $v(f_s)$를 사용하여 태양풍 속도로 매핑

In [ ]:
class SimplePFSS:
    """Simplified PFSS model with dipole + quadrupole terms.

    Solves Laplace's equation between R_sun and R_ss using low-order
    spherical harmonics. Demonstrates the key PFSS concepts:
    - Radial field at source surface
    - Open vs closed field line topology
    - Expansion factor calculation
    """

    def __init__(self, R_ss: float = 2.5, B0: float = 5.0,
                 tilt: float = 0.0, quadrupole_ratio: float = 0.0):
        """Initialize PFSS model.

        Args:
            R_ss: Source surface radius [R_sun].
            B0: Dipole field strength at poles [Gauss].
            tilt: Dipole tilt angle from rotation axis [degrees].
            quadrupole_ratio: Quadrupole to dipole strength ratio.
        """
        self.R_ss = R_ss
        self.B0 = B0
        self.tilt = np.radians(tilt)
        self.q_ratio = quadrupole_ratio

    def _radial_factor(self, r: np.ndarray, ell: int) -> np.ndarray:
        """Radial dependence for PFSS spherical harmonic of degree ell.

        For PFSS, the radial function satisfies:
        B_r ~ (ell+1)/r^(ell+2) + ell*r^(ell-1)/R_ss^(2*ell+1)
        normalized so that B_theta(R_ss) = 0.
        """
        R = self.R_ss
        # Coefficients for the potential field between R_sun and R_ss
        c1 = (ell + 1 + ell * (1.0/R)**(2*ell + 1))
        return ((ell + 1) / r**(ell + 2) +
                ell * r**(ell - 1) / R**(2*ell + 1)) / c1

    def B_r(self, r: np.ndarray, theta: np.ndarray) -> np.ndarray:
        """Compute radial magnetic field component.

        Args:
            r: Radial distance [R_sun].
            theta: Colatitude [radians].

        Returns:
            B_r [Gauss].
        """
        # Dipole (ell=1): B_r ~ cos(theta)
        Br = self.B0 * self._radial_factor(r, 1) * np.cos(theta - self.tilt)
        # Quadrupole (ell=2): B_r ~ (3cos^2(theta) - 1)/2
        if self.q_ratio != 0:
            Br += (self.B0 * self.q_ratio *
                   self._radial_factor(r, 2) *
                   0.5 * (3 * np.cos(theta)**2 - 1))
        return Br

    def compute_expansion_factor(self, theta_ss: np.ndarray) -> np.ndarray:
        """Compute flux tube expansion factor at source surface points.

        f_s = (R_sun/R_ss)^2 * |B_r(R_sun, theta_foot)| / |B_r(R_ss, theta_ss)|

        Args:
            theta_ss: Colatitudes at source surface [radians].

        Returns:
            Expansion factors (f_s >= 1).
        """
        # For a dipole, the footpoint colatitude maps approximately as:
        # sin^2(theta_foot)/R_sun = sin^2(theta_ss)/R_ss
        sin2_foot = np.sin(theta_ss)**2 / self.R_ss
        sin2_foot = np.clip(sin2_foot, 0, 1)
        theta_foot = np.arcsin(np.sqrt(sin2_foot))

        # Handle both hemispheres
        theta_foot = np.where(theta_ss > np.pi/2,
                              np.pi - theta_foot, theta_foot)

        B_r_foot = np.abs(self.B_r(1.0, theta_foot))
        B_r_ss = np.abs(self.B_r(self.R_ss, theta_ss))

        # Avoid division by zero
        B_r_ss = np.where(B_r_ss < 1e-10, 1e-10, B_r_ss)

        fs = (1.0 / self.R_ss)**2 * B_r_foot / B_r_ss
        return np.clip(fs, 1.0, None)  # f_s >= 1 by definition


# --- Create and visualize the PFSS model ---
pfss = SimplePFSS(R_ss=2.5, B0=5.0, tilt=10.0, quadrupole_ratio=0.2)

# Grid for field visualization
r_grid = np.linspace(1.0, 2.5, 50)
theta_grid = np.linspace(0.01, np.pi - 0.01, 100)
R, THETA = np.meshgrid(r_grid, theta_grid)
Br = pfss.B_r(R, THETA)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Left: B_r in meridional plane
X = R * np.sin(THETA)
Z = R * np.cos(THETA)
im = axes[0].pcolormesh(X, Z, Br, cmap='RdBu_r', vmin=-3, vmax=3,
                         shading='auto')
# Draw Sun and source surface
theta_circle = np.linspace(0, 2*np.pi, 100)
axes[0].plot(np.sin(theta_circle), np.cos(theta_circle), 'k-', linewidth=2)
axes[0].plot(2.5*np.sin(theta_circle), 2.5*np.cos(theta_circle),
             'k--', linewidth=1, alpha=0.5)
axes[0].set_xlabel('$x$ [$R_\\odot$]')
axes[0].set_ylabel('$z$ [$R_\\odot$]')
axes[0].set_title('$B_r$ in Meridional Plane\n자오면에서의 $B_r$')
axes[0].set_aspect('equal')
plt.colorbar(im, ax=axes[0], label='$B_r$ [G]')

# Middle: B_r at source surface as function of latitude
theta_ss = np.linspace(0.01, np.pi - 0.01, 200)
lat_ss = 90 - np.degrees(theta_ss)  # Convert to latitude
Br_ss = pfss.B_r(2.5, theta_ss)
axes[1].plot(lat_ss, Br_ss, 'b-', linewidth=2)
axes[1].axhline(y=0, color='k', linestyle='-', alpha=0.3)
axes[1].fill_between(lat_ss, Br_ss, 0, where=Br_ss > 0,
                      alpha=0.2, color='red', label='Outward (+)')
axes[1].fill_between(lat_ss, Br_ss, 0, where=Br_ss < 0,
                      alpha=0.2, color='blue', label='Inward (−)')
axes[1].set_xlabel('Latitude [deg]')
axes[1].set_ylabel('$B_r$ at Source Surface [G]')
axes[1].set_title('$B_r$ at Source Surface ($r = 2.5 R_\\odot$)\nSource Surface에서의 $B_r$')
axes[1].legend()

# Right: Expansion factor and resulting velocity
fs_vals = pfss.compute_expansion_factor(theta_ss)
v_vals = wsa_velocity(fs_vals)

ax_fs = axes[2]
ax_v = ax_fs.twinx()
l1, = ax_fs.semilogy(lat_ss, fs_vals, 'b-', linewidth=2, label='$f_s$')
l2, = ax_v.plot(lat_ss, v_vals, 'r-', linewidth=2, label='$v$ [km/s]')
ax_fs.set_xlabel('Latitude [deg]')
ax_fs.set_ylabel('Expansion Factor $f_s$', color='b')
ax_v.set_ylabel('Solar Wind Speed $v$ [km/s]', color='r')
axes[2].set_title('Expansion Factor & Velocity\n팽창 계수 및 속도')
ax_fs.legend(handles=[l1, l2], loc='upper right')

plt.tight_layout()
plt.show()

# Print key values at specific latitudes
print("=== PFSS Results at Selected Latitudes / 선택 위도에서의 PFSS 결과 ===")
print(f"{'Lat [°]':>8s}  {'B_r(R_ss) [G]':>14s}  {'f_s':>8s}  {'v [km/s]':>10s}")
print("-" * 50)
for lat_deg in [80, 60, 40, 20, 0, -20, -40, -60, -80]:
    theta = np.radians(90 - lat_deg)
    br = pfss.B_r(2.5, theta)
    fs = pfss.compute_expansion_factor(np.array([theta]))[0]
    v = wsa_velocity(fs)
    print(f"{lat_deg:>8d}  {br:>14.4f}  {fs:>8.1f}  {v:>10.1f}")

## Part 6: Full WSA Pipeline — End-to-End Demonstration / 전체 WSA 파이프라인

Now we connect all components into a complete WSA pipeline: PFSS → $f_s$ → $v(f_s)$ → stream propagation → velocity at Earth.

이제 모든 구성 요소를 완전한 WSA 파이프라인으로 연결합니다: PFSS → $f_s$ → $v(f_s)$ → 스트림 전파 → 지구에서의 속도.

We also compute the statistical metrics used in the paper: **Average Fractional Deviation (AFD)** and **Correlation Coefficient (CC)**.

논문에서 사용된 통계 지표도 계산합니다: **평균 분수 편차 (AFD)**와 **상관 계수 (CC)**.

In [ ]:
def compute_afd(v_pred: np.ndarray, v_obs: np.ndarray) -> float:
    """Compute Average Fractional Deviation.

    AFD = <|v_pred - v_obs| / v_obs>

    Args:
        v_pred: Predicted velocities [km/s].
        v_obs: Observed velocities [km/s].

    Returns:
        AFD (dimensionless). Typical good values: 0.10-0.15.
    """
    return np.mean(np.abs(v_pred - v_obs) / v_obs)


def compute_cc(v_pred: np.ndarray, v_obs: np.ndarray) -> float:
    """Compute Pearson correlation coefficient.

    Args:
        v_pred: Predicted velocities.
        v_obs: Observed velocities.

    Returns:
        Correlation coefficient [-1, 1].
    """
    return np.corrcoef(v_pred, v_obs)[0, 1]


# === Full pipeline demonstration ===
np.random.seed(42)

# Step 1: PFSS model (tilted dipole + weak quadrupole)
pfss_model = SimplePFSS(R_ss=2.5, B0=5.0, tilt=15.0, quadrupole_ratio=0.3)

# Step 2: Sample expansion factors along ecliptic
# The ecliptic samples different colatitudes at source surface
# depending on the Carrington longitude (due to dipole tilt)
n_cells = 72
carr_lon = np.linspace(0, 360, n_cells, endpoint=False)

# Ecliptic colatitude varies sinusoidally around 90° due to tilt
ecliptic_colat = np.radians(90 - 15 * np.sin(np.radians(carr_lon)))

# Step 3: Compute expansion factors
fs_ecliptic = pfss_model.compute_expansion_factor(ecliptic_colat)

# Step 4: Convert to velocity
v_source_full = wsa_velocity(fs_ecliptic)

# Step 5: Set release times (Sun rotation ~13.2°/day)
release_t = carr_lon / 13.2

# Step 6: Propagate to Earth
t_earth_full, v_earth_full = propagate_to_earth(v_source_full, release_t)

# Step 7: Generate synthetic "observed" data with noise and CME contamination
# Resample predicted velocities onto a regular time grid
t_regular = np.linspace(t_earth_full.min(), t_earth_full.max(), 100)
v_pred_interp = np.interp(t_regular, t_earth_full, v_earth_full)

# Simulated observations: predicted + noise + occasional CME spikes
noise = np.random.normal(0, 30, len(t_regular))  # ±30 km/s noise
v_obs_sim = v_pred_interp + noise

# Add 2 CME events (large speed enhancements the model can't predict)
cme_mask1 = (t_regular > 12) & (t_regular < 13)
v_obs_sim[cme_mask1] += 200
cme_mask2 = (t_regular > 22) & (t_regular < 22.5)
v_obs_sim[cme_mask2] += 150
v_obs_sim = np.clip(v_obs_sim, 250, 900)

# Step 8: Compute statistics
afd = compute_afd(v_pred_interp, v_obs_sim)
cc = compute_cc(v_pred_interp, v_obs_sim)

# Exclude CME periods
no_cme = ~(cme_mask1 | cme_mask2)
afd_no_cme = compute_afd(v_pred_interp[no_cme], v_obs_sim[no_cme])
cc_no_cme = compute_cc(v_pred_interp[no_cme], v_obs_sim[no_cme])

# --- Comprehensive visualization ---
fig = plt.figure(figsize=(16, 12))
gs = GridSpec(3, 2, figure=fig, hspace=0.35)

# (a) Expansion factor profile along ecliptic
ax1 = fig.add_subplot(gs[0, 0])
ax1.semilogy(carr_lon, fs_ecliptic, 'k-', linewidth=2)
ax1.set_xlabel('Carrington Longitude [deg]')
ax1.set_ylabel('$f_s$ (log scale)')
ax1.set_title('(a) Expansion Factor Along Ecliptic\n     황도면을 따른 팽창 계수')
ax1.axhspan(1, 5, alpha=0.1, color='blue')

# (b) Velocity at source surface
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(carr_lon, v_source_full, 'b-', linewidth=2)
ax2.set_xlabel('Carrington Longitude [deg]')
ax2.set_ylabel('$v$ [km/s]')
ax2.set_title('(b) Velocity at Source Surface\n     Source Surface에서의 속도')
ax2.set_ylim(250, 700)

# (c) Predicted vs "Observed" at Earth
ax3 = fig.add_subplot(gs[1, :])
ax3.plot(t_regular, v_obs_sim, 'k-', linewidth=1, alpha=0.8,
         label='Observed (synthetic) / 관측 (합성)')
ax3.plot(t_regular, v_pred_interp, 'r-', linewidth=2,
         label='WSA Predicted / WSA 예측')
# Mark CME events
ax3.axvspan(12, 13, alpha=0.2, color='yellow', label='CME events')
ax3.axvspan(22, 22.5, alpha=0.2, color='yellow')
ax3.set_xlabel('Time [days]')
ax3.set_ylabel('Solar Wind Speed [km/s]')
ax3.set_title(f'(c) WSA Prediction vs Observation at Earth '
              f'(CC={cc:.3f}, AFD={afd:.3f})\n'
              f'     WSA 예측 대 지구 관측 '
              f'(CME 제외: CC={cc_no_cme:.3f}, AFD={afd_no_cme:.3f})')
ax3.legend(loc='upper right')
ax3.set_ylim(250, 800)

# (d) Scatter plot: predicted vs observed
ax4 = fig.add_subplot(gs[2, 0])
ax4.scatter(v_obs_sim[no_cme], v_pred_interp[no_cme], alpha=0.5,
            c='blue', s=20, label='Background wind')
ax4.scatter(v_obs_sim[~no_cme], v_pred_interp[~no_cme], alpha=0.8,
            c='red', s=40, marker='x', label='CME periods')
ax4.plot([250, 800], [250, 800], 'k--', alpha=0.5, label='Perfect prediction')
ax4.set_xlabel('Observed $v$ [km/s]')
ax4.set_ylabel('Predicted $v$ [km/s]')
ax4.set_title('(d) Scatter: Predicted vs Observed\n     산점도: 예측 대 관측')
ax4.legend()
ax4.set_aspect('equal')
ax4.set_xlim(250, 800)
ax4.set_ylim(250, 800)

# (e) Residuals
ax5 = fig.add_subplot(gs[2, 1])
residuals = v_pred_interp - v_obs_sim
ax5.plot(t_regular, residuals, 'b-', linewidth=1, alpha=0.7)
ax5.axhline(y=0, color='k', linestyle='-', alpha=0.3)
ax5.fill_between(t_regular, -60, 60, alpha=0.1, color='green',
                  label='±60 km/s (AFD~0.15)')
ax5.set_xlabel('Time [days]')
ax5.set_ylabel('Residual $v_{pred} - v_{obs}$ [km/s]')
ax5.set_title('(e) Prediction Residuals\n     예측 잔차')
ax5.legend()

plt.tight_layout()
plt.show()

# Print summary statistics
print("=" * 60)
print("WSA Pipeline Summary Statistics / 파이프라인 요약 통계")
print("=" * 60)
print(f"  All data:       CC = {cc:.3f},  AFD = {afd:.3f}")
print(f"  Excl. CMEs:     CC = {cc_no_cme:.3f},  AFD = {afd_no_cme:.3f}")
print(f"  Paper reports:  CC ~ 0.39,  AFD ~ 0.15 (MDU method)")
print(f"  Mean residual (no CME): {residuals[no_cme].mean():+.1f} km/s")
print(f"  Std residual (no CME):  {residuals[no_cme].std():.1f} km/s")

## Summary / 요약

| Concept / 개념 | This Paper / 이 논문 | Modern Equivalent / 현대 동등물 |
|---|---|---|
| Coronal field model / 코로나 자기장 모델 | PFSS with $\ell_{max} = 30$ from WSO data | `pfsspy` package with SDO/HMI synoptic maps |
| $f_s$ → $v$ mapping / 매핑 | $v = 267.5 + 410/f_s^{2/5}$ (source surface) | WSA with $\theta_b$ parameter (Arge et al. 2004) |
| Stream propagation / 스트림 전파 | 1/8 AU radial steps + harmonic merge | ENLIL 3D MHD heliospheric model |
| Data quality control / 데이터 품질 관리 | $Q$ factor from polar field properties | Automated QC pipelines (GONG, SDO/HMI) |
| LOS → $B_r$ correction / 보정 | $B_l / [\sin\theta \cos(\phi-\phi_o)]$ | Vector magnetograms (no LOS correction needed) |
| Synoptic map update / 싱옵틱 맵 갱신 | Daily updated from WSO | Near-real-time from GONG (6 sites worldwide) |

### Key Implementation Insights / 핵심 구현 시사점

1. **$v(f_s)$ is remarkably simple** — a two-parameter empirical function that captures the essential physics of coronal hole → fast wind mapping. Its simplicity enables real-time operational use.
$v(f_s)$는 놀라울 정도로 단순합니다 — coronal hole → 고속풍 매핑의 핵심 물리학을 포착하는 2-매개변수 경험 함수. 단순함이 실시간 운용 사용을 가능하게 합니다.

2. **Stream interaction modeling matters** — even the simple 1/8 AU step merge scheme produces physically meaningful velocity profile evolution. Fast streams steepen and slow streams compress, mimicking CIR formation.
스트림 상호작용 모델링이 중요합니다 — 단순한 1/8 AU 단계 병합도 물리적으로 의미 있는 속도 프로파일 진화를 생성합니다.

3. **Data quality is the bottleneck** — the Q factor and polar field corrections are what separate MDU from DU performance. This lesson extends to modern implementations: magnetogram quality directly limits forecast skill.
데이터 품질이 병목입니다 — Q factor와 극 자기장 보정이 MDU와 DU 성능을 구분합니다. 이 교훈은 현대 구현에도 적용됩니다.